//////// Scraping API ///////

In [12]:
import requests
import pandas as pd

url = "https://opensky-network.org/api/states/all"

response = requests.get(url)
data = response.json()

print("Status :", response.status_code)
print("Nombre d'observations :", len(data["states"]))

Status : 200
Nombre d'observations : 12067


/////////// Phase exploratoire ///////////

In [13]:
columns = [
    "icao24",
    "callsign",
    "origin_country",
    "time_position",
    "last_contact",
    "longitude",
    "latitude",
    "baro_altitude",
    "on_ground",
    "velocity",
    "true_track",
    "vertical_rate",
    "sensors",
    "geo_altitude",
    "squawk",
    "spi",
    "position_source"
]

df = pd.DataFrame(data["states"], columns=columns)

df.head()

,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source
0,39de4f,TVF37MF,France,1.790775e+09,1790774952,4.6121,46.6761,11277.60,False,238.43,124.27,0.33,None,11818.62,1000,False,0
1,a5f852,RTY484,United States,1.790775e+09,1790774952,-104.7548,41.2279,2872.74,False,59.38,37.96,-0.65,None,2903.22,None,False,0
2,ab1644,UAL214,United States,1.790775e+09,1790774952,-99.8164,32.4044,10363.20,False,248.39,328.12,0.33,None,11010.90,None,False,0
3,e8027e,LPE2464,Chile,1.790775e+09,1790774821,-63.7331,-28.2150,10980.42,False,180.38,323.82,0.00,None,11430.00,None,False,0
4,aa3cbe,,United States,1.790775e+09,1790774840,-111.8515,41.7812,NaN,True,3.09,36.56,NaN,None,NaN,None,False,0


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12067 entries, 0 to 12066
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   icao24           12067 non-null  object 
 1   callsign         12067 non-null  object 
 2   origin_country   12067 non-null  object 
 3   time_position    11974 non-null  float64
 4   last_contact     12067 non-null  int64  
 5   longitude        11974 non-null  float64
 6   latitude         11974 non-null  float64
 7   baro_altitude    10867 non-null  float64
 8   on_ground        12067 non-null  bool   
 9   velocity         12060 non-null  float64
 10  true_track       12067 non-null  float64
 11  vertical_rate    10880 non-null  float64
 12  sensors          0 non-null      object 
 13  geo_altitude     10776 non-null  float64
 14  squawk           6594 non-null   object 
 15  spi              12067 non-null  bool   
 16  position_source  12067 non-null  int64  
dtypes: bool(2), 

In [15]:
df.isnull().sum()

icao24                 0
callsign               0
origin_country         0
time_position         93
last_contact           0
longitude             93
latitude              93
baro_altitude       1200
on_ground              0
velocity               7
true_track             0
vertical_rate       1187
sensors            12067
geo_altitude        1291
squawk              5473
spi                    0
position_source        0
dtype: int64

In [17]:
(df.isnull().mean() * 100).round(2)

icao24               0.00
callsign             0.00
origin_country       0.00
time_position        0.77
last_contact         0.00
longitude            0.77
latitude             0.77
baro_altitude        9.94
on_ground            0.00
velocity             0.06
true_track           0.00
vertical_rate        9.84
sensors            100.00
geo_altitude        10.70
squawk              45.36
spi                  0.00
position_source      0.00
dtype: float64

In [18]:
df.duplicated().sum()

np.int64(0)

In [19]:
df["icao24"].duplicated().sum()

np.int64(0)

////////////// Nettoyage des données //////////////////

supression colomne avec 100% valeurs nulles

In [20]:
df = df.drop(columns=["sensors"])

In [21]:
df.shape

(12067, 16)

nettoyage nom indicatif de vol (suppression des espaces inutiles)

In [22]:
df["callsign"] = df["callsign"].str.strip()

In [23]:
df["callsign"].head(10)

0    TVF37MF
1     RTY484
2     UAL214
3    LPE2464
4           
5    LPE2290
6    AXB2945
7    TVF27CR
8    TVF28HN
9    TVF8939
Name: callsign, dtype: object

conversion dates Unix (seconde depuis 1970) vers date UTC (vrais dates/heures)

In [24]:
df["time_position"] = pd.to_datetime(df["time_position"], unit="s", utc=True)
df["last_contact"] = pd.to_datetime(df["last_contact"], unit="s", utc=True)

In [25]:
df[["time_position", "last_contact"]].head()

,time_position,last_contact
0,2026-09-30 13:29:12+00:00,2026-09-30 13:29:12+00:00
1,2026-09-30 13:29:12+00:00,2026-09-30 13:29:12+00:00
2,2026-09-30 13:29:12+00:00,2026-09-30 13:29:12+00:00
3,2026-09-30 13:27:01+00:00,2026-09-30 13:27:01+00:00
4,2026-09-30 13:27:20+00:00,2026-09-30 13:27:20+00:00


contrôle cohérence coordonnées GPS (latitude/longitude)

In [26]:
invalid_coordinates = df[
    (df["latitude"] < -90) |
    (df["latitude"] > 90) |
    (df["longitude"] < -180) |
    (df["longitude"] > 180)
]

len(invalid_coordinates)

0

Cohérence des valeurs

cohérence de la velocité (m/s)

In [27]:
df["velocity"].describe()

count    12060.000000
mean       149.997468
std         91.714934
min          0.000000
25%         60.517500
50%        177.985000
75%        229.840000
max       1606.400000
Name: velocity, dtype: float64

In [30]:
df.nlargest(10, "velocity")[
    ["icao24", "callsign", "origin_country", "velocity", "on_ground", "baro_altitude"]
]

,icao24,callsign,origin_country,velocity,on_ground,baro_altitude
8812,49d4cf,QAV30E,Czech Republic,1606.40,False,563.88
8065,4baa62,THY3CE,Turkey,1479.39,False,10668.00
840,4bc883,PGT397,Turkey,1320.97,False,10668.00
11049,4c0222,ASL987,Serbia,1311.82,False,11582.40
5126,503c24,SZN4002,Lithuania,706.38,False,11582.40
2755,461f08,FIN122,Finland,530.45,False,11887.20
10988,45d962,VKG347,Denmark,519.60,False,3688.08
3683,4241e2,VPCYW,United Kingdom,385.05,False,11269.98
4611,503e99,GJT7KL,Lithuania,340.21,False,11277.60
9571,010282,LMU616,Egypt,329.05,False,10058.40


In [31]:
df[df["velocity"] > 400][
    ["icao24", "callsign", "origin_country", "velocity", "baro_altitude"]
]

,icao24,callsign,origin_country,velocity,baro_altitude
840,4bc883,PGT397,Turkey,1320.97,10668.00
2755,461f08,FIN122,Finland,530.45,11887.20
5126,503c24,SZN4002,Lithuania,706.38,11582.40
8065,4baa62,THY3CE,Turkey,1479.39,10668.00
8812,49d4cf,QAV30E,Czech Republic,1606.40,563.88
10988,45d962,VKG347,Denmark,519.60,3688.08
11049,4c0222,ASL987,Serbia,1311.82,11582.40


altitude géométrique (GPS)

In [32]:
df["baro_altitude"].describe()

count    10867.000000
mean      6306.624288
std       4473.073372
min       -137.160000
25%       1459.230000
50%       7010.400000
75%      10668.000000
max      31973.520000
Name: baro_altitude, dtype: float64

In [33]:
df.nlargest(10, "baro_altitude")[
    ["icao24", "callsign", "origin_country", "baro_altitude", "velocity"]
]

,icao24,callsign,origin_country,baro_altitude,velocity
2471,01808f,LWA103,Libyan Arab Jamahiriya,31973.52,223.28
4059,a26454,HBAL815,United States,19751.04,0.73
11211,a09af1,N138WV,United States,17007.84,3.00
8551,ad08d7,KOW939,United States,14325.60,247.04
8597,a6ffcc,N550DX,United States,14325.60,237.37
579,addb2b,KOW992,United States,14135.10,236.87
8503,a82d60,N626JS,United States,13723.62,250.55
330,4520d3,EGT525,Bulgaria,13716.00,218.28
417,a27ccf,N26BT,United States,13716.00,261.63
451,a27cbd,N26B,United States,13716.00,225.07


altitude barométrique (pression air)

In [34]:
df["geo_altitude"].describe()

count    10776.000000
mean      6603.578560
std       4641.483627
min        -53.340000
25%       1546.860000
50%       7383.780000
75%      11117.580000
max      20711.160000
Name: geo_altitude, dtype: float64

In [40]:
df.nlargest(10, "geo_altitude")[
    ["icao24", "callsign", "baro_altitude", "geo_altitude", "velocity"]
]

,icao24,callsign,baro_altitude,geo_altitude,velocity
4059,a26454,HBAL815,19751.04,20711.16,0.73
11211,a09af1,N138WV,17007.84,17967.96,3.00
8551,ad08d7,KOW939,14325.60,14988.54,247.04
8597,a6ffcc,N550DX,14325.60,14958.06,237.37
579,addb2b,KOW992,14135.10,14668.50,236.87
4903,a496a5,POD5,13716.00,14478.00,240.89
2709,a4fc32,LXJ420,13716.00,14470.38,224.12
8944,a08f33,EJA135,13716.00,14455.14,259.69
4296,a659be,VJA508,13716.00,14447.52,216.19
4029,a6cb01,LXJ537,13708.38,14439.90,212.00


vitesse verticale

In [41]:
df["vertical_rate"].describe()

count    10880.000000
mean         0.088169
std          5.068909
min       -165.810000
25%         -0.650000
50%          0.000000
75%          0.330000
max         39.990000
Name: vertical_rate, dtype: float64

In [42]:
df.nsmallest(10, "vertical_rate")[
    ["icao24", "callsign", "vertical_rate",
     "baro_altitude", "velocity"]
]

,icao24,callsign,vertical_rate,baro_altitude,velocity
7521,403d79,GILDA,-165.81,647.70,266.64
9443,39514a,MOLOCH66,-28.61,1417.32,84.48
423,ae6268,ROKT99,-28.29,2948.94,123.61
7273,00ae17,SFR321,-21.78,4953.00,199.30
8600,4ca763,RYR81PR,-21.46,7993.38,232.81
4501,0d0939,VIV8010,-20.81,7543.80,223.02
4271,48e9f3,RYR6GW,-20.16,5036.82,177.27
5820,06a10b,QTR124,-20.16,7338.06,230.49
2890,76cd16,SIA921,-19.83,10005.06,266.49
9562,406445,EZY36DH,-19.83,10248.90,222.59


In [43]:
df.nlargest(10, "vertical_rate")[
    ["icao24", "callsign", "vertical_rate",
     "baro_altitude", "velocity"]
]

,icao24,callsign,vertical_rate,baro_altitude,velocity
1348,ae1ed5,BB080,39.99,4267.20,105.40
5409,ac8a7b,EJM907,29.59,1356.36,128.45
6359,a03159,EJA111,29.26,1562.10,135.31
11650,a4ddf6,N4120R,28.94,487.68,64.31
12050,39d517,FHVIX,28.94,2476.50,95.45
10701,4d2330,FJO54A,25.36,441.96,78.63
9013,49d675,ELI30B,24.71,2362.20,116.48
5598,440405,MOZ437,23.41,2065.02,101.88
5925,aca5a4,N914BD,23.41,2362.20,138.96
8141,43c8bb,PRF44,23.41,1676.40,49.14


direction de déplacement (en degré °)

In [44]:
df["true_track"].describe()

count    12067.000000
mean       185.416895
std        102.287192
min          0.000000
25%         95.760000
50%        191.040000
75%        273.935000
max        359.870000
Name: true_track, dtype: float64

Création de df_clean

In [45]:
df_clean = df.copy()

In [46]:
df_clean = df_clean.drop(columns=["sensors"], errors="ignore")

In [47]:
df_clean = df_clean.dropna(subset=["latitude", "longitude"])

In [49]:
print("RAW :", len(df))
print("CLEAN :", len(df_clean))
print("Supprimées :", len(df) - len(df_clean))

RAW : 12067
CLEAN : 11974
Supprimées : 93


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12067 entries, 0 to 12066
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   icao24           12067 non-null  object             
 1   callsign         12067 non-null  object             
 2   origin_country   12067 non-null  object             
 3   time_position    11974 non-null  datetime64[ns, UTC]
 4   last_contact     12067 non-null  datetime64[ns, UTC]
 5   longitude        11974 non-null  float64            
 6   latitude         11974 non-null  float64            
 7   baro_altitude    10867 non-null  float64            
 8   on_ground        12067 non-null  bool               
 9   velocity         12060 non-null  float64            
 10  true_track       12067 non-null  float64            
 11  vertical_rate    10880 non-null  float64            
 12  geo_altitude     10776 non-null  float64            
 13  squawk          

In [51]:
(df_clean["velocity"] < 0).sum()

np.int64(0)

In [52]:
(df_clean["baro_altitude"] < 0).sum()

np.int64(45)

In [56]:
df_clean[df_clean["velocity"] > 400][
    ["icao24", "callsign", "velocity", "baro_altitude"]
]

,icao24,callsign,velocity,baro_altitude


In [54]:
df_clean.loc[df_clean["velocity"] > 400, "velocity"] = pd.NA

In [57]:
(df_clean["velocity"] > 400).sum()

np.int64(0)

In [58]:
df_clean[df_clean["baro_altitude"] > 20000][
    ["icao24", "callsign", "baro_altitude", "geo_altitude", "velocity"]
]

,icao24,callsign,baro_altitude,geo_altitude,velocity
2471,01808f,LWA103,31973.52,11902.44,223.28


In [59]:
df_clean.loc[df_clean["baro_altitude"] > 20000, "baro_altitude"] = pd.NA

In [60]:
(df_clean["baro_altitude"] > 20000).sum()

np.int64(0)

In [61]:
df_clean[df_clean["geo_altitude"] > 20000][
    ["icao24", "callsign", "baro_altitude", "geo_altitude", "velocity"]
]

,icao24,callsign,baro_altitude,geo_altitude,velocity
4059,a26454,HBAL815,19751.04,20711.16,0.73


In [62]:
df_clean[df_clean["vertical_rate"].abs() > 50][
    ["icao24", "callsign", "vertical_rate", "baro_altitude", "velocity"]
]

,icao24,callsign,vertical_rate,baro_altitude,velocity
7521,403d79,GILDA,-165.81,647.7,266.64


In [63]:
df_clean.loc[df_clean["vertical_rate"].abs() > 50, "vertical_rate"] = pd.NA

In [64]:
(df_clean["vertical_rate"].abs() > 50).sum()

np.int64(0)

///// creation du dataframe nettoyé TEST ////

Création du dataset CLEAN

In [ ]:
df_test = df.copy()

print("Nombre de lignes au départ :", len(df_test))

Nombre de lignes RAW : 12067


Suppression des colonnes inutilisables

In [ ]:
empty_columns = df_test.columns[df_test.isna().all()].tolist()
print("Colonnes entièrement vides :", empty_columns)

df_test = df_test.drop(columns=empty_columns)

Colonnes entièrement vides : []


Nettoyage des textes

In [ ]:
text_columns = ["icao24", "callsign", "origin_country", "squawk"]

for column in text_columns:
    if column in df_test.columns:
        df_test[column] = df_test[column].str.strip()

In [ ]:
print((df_clean["icao24", "callsign", "origin_country", "squawk"].head))